In [37]:
import duckdb

con = duckdb.connect()

path = "../../source_database/seed_data/order_items.csv"

### 1.Schema audit

In [3]:
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_item_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ unit_price    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ line_total    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ discount_pct  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ created_at    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



### 2. Completeness audit

In [4]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 5310


,column_name,null_count,null_percentage
0,order_item_id,0,0.0
1,order_id,0,0.0
2,product_id,0,0.0
3,quantity,0,0.0
4,unit_price,0,0.0
5,line_total,0,0.0
6,discount_pct,0,0.0
7,created_at,0,0.0


### 3. Uniqueness / primary key audit

In [6]:
# ── Uniqueness Audit: does order_id actually behave like a primary key? ──

# Step 1: Group all rows by order_id, and count how many rows fall into each group.
#   - If order_id is truly unique, every group will have exactly 1 row.
#   - HAVING COUNT(*) > 1 keeps only the groups with MORE than one row —
#     i.e. the duplicate order_ids.
duplicates_df = con.sql(f"""
    SELECT
        order_item_id,
        COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY order_item_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
""").df()

print(f"Number of duplicate order_item_id values found: {len(duplicates_df)}")
duplicates_df

Number of duplicate order_item_id values found: 0


,order_item_id,row_count


### 4. Referential integrity audit

In [12]:
# ── Referential Integrity Audit: does every order_items's customer_id exist in customers.csv? ──

orders_path = "../../source_database/seed_data/orders.csv"
products_path = "../../source_database/seed_data/products.csv"

orphan_orders = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('{orders_path}') s ON i.order_id = s.order_id
    WHERE s.order_id IS NULL
""").fetchone()[0]
print(f"Orphaned order_id rows: {orphan_orders}")

orphan_products = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('{products_path}') p ON i.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"Orphaned product_id rows: {orphan_products}")

Orphaned order_id rows: 0
Orphaned product_id rows: 0


### 5. Validity / business-rule audit

In [16]:
# Negative money values — a sale should never have a negative amount
negative_amounts_df = con.sql(f"""
    SELECT order_id, product_id, quantity, unit_price, line_total
    FROM read_csv_auto('{path}')
    WHERE order_id < 0 OR product_id < 0 OR quantity < 0 OR unit_price < 0 OR line_total < 0
""").df()
print(f"Order_items with a negative amount field: {len(negative_amounts_df)}")

Order_items with a negative amount field: 0


In [28]:
# discount_pct must be a real percentage (0 to 1)

# WHY: discount_pct is supposed to be a fraction (0.05 = 5% off). A value
# outside [0,1] means either a formatting bug

bad_discount = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}') WHERE discount_pct < 0 OR discount_pct > 1
""").df()
print(f"discount_pct outside [0,1]: {len(bad_discount)} rows")
print(bad_discount.head(10))

discount_pct outside [0,1]: 0 rows
Empty DataFrame
Columns: [order_item_id, order_id, product_id, quantity, unit_price, line_total, discount_pct, created_at]
Index: []


In [30]:
# quantity should be a whole number

# WHY: order line quantities are normally whole units ("3 shirts"), not
# fractions. A fractional quantity (2.9, 3.7...) suggests either a
# unit-of-measure mix-up (kg/litres mixed in with piece counts) or a
# random-data-generation bug. Not technically invalid, but worth a look.

frac_qty = con.sql(f"""
    SELECT order_item_id, order_id, product_id, quantity FROM read_csv_auto('{path}') WHERE quantity != ROUND(quantity)
""").df()
print(f"non-integer quantity: {len(frac_qty)} rows")
print(frac_qty.head(10))

non-integer quantity: 1391 rows
   order_item_id  order_id  product_id  quantity
0              1         1          41       2.9
1              6         1          39       2.4
2              9         2         110       3.7
3             10         2         138       3.6
4             22         6          77       2.7
5             23         7          29       2.8
6             26         9           9       2.6
7             27         9         147       2.7
8             28        10         150       4.6
9             29        10          70       5.5


In [35]:
# does line_total actually apply the discount?

# WHY: this is the main one. line_total should equal quantity x unit_price x
# (1 - discount_pct). If it instead equals quantity x unit_price with the
# discount ignored, every revenue figure built on top of this table is
# overstated by the discount amount — silently.

line_total_check = con.sql(f"""
    SELECT order_item_id, quantity, unit_price, discount_pct, line_total,
           ROUND(CAST(ROUND(quantity) AS INTEGER) * unit_price * (1 - discount_pct), 2) AS expected_with_discount,
           ROUND(CAST(ROUND(quantity) AS INTEGER) * unit_price, 2) AS expected_without_discount,
           CASE
               WHEN ABS(line_total - ROUND(CAST(ROUND(quantity) AS INTEGER) * unit_price * (1 - discount_pct), 2)) < 0.01
                   THEN 'matches WITH discount'
               WHEN ABS(line_total - ROUND(CAST(ROUND(quantity) AS INTEGER) * unit_price, 2)) < 0.01
                   THEN 'matches WITHOUT discount'
               ELSE 'matches NEITHER'
           END AS pattern
    FROM read_csv_auto('{path}')
""").df()

print(line_total_check['pattern'].value_counts())
print("\nRows matching neither formula (genuine calc errors):")
print(line_total_check[line_total_check['pattern'] == 'matches NEITHER'].head(10))

pattern
matches WITHOUT discount    3793
matches NEITHER             1364
matches WITH discount        153
Name: count, dtype: int64

Rows matching neither formula (genuine calc errors):
    order_item_id  quantity  unit_price  discount_pct  line_total  \
0               1       2.9       11.91          0.04       34.54   
5               6       2.4        1.44          0.10        3.46   
8               9       3.7        5.25          0.02       19.43   
9              10       3.6        7.29          0.04       26.24   
21             22       2.7        7.09          0.08       19.14   
22             23       2.8       10.98          0.11       30.74   
25             26       2.6        2.45          0.12        6.37   
26             27       2.7        8.76          0.09       23.65   
27             28       4.6        2.74          0.03       12.60   
28             29       5.5        5.32          0.07       29.26   

    expected_with_discount  expected_without_discount

In [34]:
# WHY: this is the actual business formula the user specified:
# Line total = (Unit Price * Quantity) * (1 - Discount Percentage)
# quantity is cast to an integer (rounded) first, since the data has some
# fractional quantities (2.9, 3.7, ...) that are a smell, not the real count.

formula_check = con.sql(f"""
    SELECT
        order_item_id,
        CAST(ROUND(quantity) AS INTEGER) AS quantity_int,
        unit_price,
        discount_pct,
        line_total,
        ROUND(unit_price * CAST(ROUND(quantity) AS INTEGER) * (1 - discount_pct), 2) AS calculated_line_total,
        CASE
            WHEN ABS(line_total - ROUND(unit_price * CAST(ROUND(quantity) AS INTEGER) * (1 - discount_pct), 2)) < 0.01
                THEN 'MATCH'
            ELSE 'MISMATCH'
        END AS result
    FROM read_csv_auto('{path}')
""").df()

print(formula_check['result'].value_counts())

print("\n=== Mismatches ===")
print(formula_check[formula_check['result'] == 'MISMATCH'])

result
MISMATCH    5157
MATCH        153
Name: count, dtype: int64

=== Mismatches ===
      order_item_id  quantity_int  unit_price  discount_pct  line_total  \
0                 1             3       11.91          0.04       34.54   
1                 2             3        8.17          0.05       24.51   
2                 3             6        8.55          0.06       51.30   
3                 4             3        9.10          0.02       27.30   
4                 5             6        9.18          0.06       55.08   
...             ...           ...         ...           ...         ...   
5305           5306             2        4.96          0.01        9.42   
5306           5307             2        6.90          0.07       13.80   
5307           5308             1        6.57          0.04        6.57   
5308           5309             4        1.43          0.02        5.72   
5309           5310             2        4.69          0.07        8.44   

      calcul

In [38]:
formula_check = con.sql(f"""
    SELECT
        order_item_id,
        CAST(quantity AS INTEGER) AS quantity_int,
        unit_price,
        discount_pct,
        line_total,
        ROUND(unit_price * CAST(quantity AS INTEGER) * (1 - discount_pct), 2) AS calculated_line_total,
        CASE
            WHEN ABS(line_total - ROUND(unit_price * CAST(quantity AS INTEGER) * (1 - discount_pct), 2)) < 0.01
                THEN 'MATCH'
            ELSE 'MISMATCH'
        END AS result
    FROM read_csv_auto('{path}')
""").df()

print(formula_check['result'].value_counts())

print("\n=== Mismatches ===")
print(formula_check[formula_check['result'] == 'MISMATCH'])

result
MISMATCH    5159
MATCH        151
Name: count, dtype: int64

=== Mismatches ===
      order_item_id  quantity_int  unit_price  discount_pct  line_total  \
0                 1             3       11.91          0.04       34.54   
1                 2             3        8.17          0.05       24.51   
2                 3             6        8.55          0.06       51.30   
3                 4             3        9.10          0.02       27.30   
4                 5             6        9.18          0.06       55.08   
...             ...           ...         ...           ...         ...   
5305           5306             2        4.96          0.01        9.42   
5306           5307             2        6.90          0.07       13.80   
5307           5308             1        6.57          0.04        6.57   
5308           5309             4        1.43          0.02        5.72   
5309           5310             2        4.69          0.07        8.44   

      calcul

In [17]:
# 6. Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,order_item_id,INTEGER,5310,False
1,order_id,INTEGER,5310,False
2,product_id,INTEGER,5310,False
3,quantity,INTEGER,5310,False
4,unit_price,INTEGER,5310,False
5,line_total,INTEGER,5310,False
6,discount_pct,INTEGER,5310,False
7,created_at,DATE,5310,False


In [18]:
# 7. Sentinel/Default value usage:

# scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count
0,order_item_id,999,1
1,order_id,999,2


In [20]:
# 8. DISTRIBUTION / OUTLIER (IQR)

def iqr_outliers(con, path, col):
    return con.sql(f"""
        WITH stats AS (
            SELECT QUANTILE_CONT({col}, 0.25) AS q1,
                   QUANTILE_CONT({col}, 0.75) AS q3
            FROM read_csv_auto('{path}')
        )
        SELECT t.order_item_id, t.{col}
        FROM read_csv_auto('{path}') t, stats
        WHERE t.{col} < (stats.q1 - 1.5*(stats.q3-stats.q1))
           OR t.{col} > (stats.q3 + 1.5*(stats.q3-stats.q1))
        ORDER BY t.{col} DESC
    """).df()

print("\n OUTLIERS: line_total ===")
print(iqr_outliers(con, path, "line_total").head(15))
print("\n OUTLIERS: unit_price ===")
print(iqr_outliers(con, path, "unit_price").head(15))


 OUTLIERS: line_total ===
    order_item_id  line_total
0             738       71.88
1            3381       71.88
2            4022       71.52
3            2853       71.34
4            2998       71.28
5            4883       71.28
6            1978       71.22
7            4078       71.22
8            1199       71.16
9            3174       71.10
10           4001       71.04
11           2547       70.98
12           5303       70.86
13           3332       70.80
14           4941       70.56

 OUTLIERS: unit_price ===
Empty DataFrame
Columns: [order_item_id, unit_price]
Index: []


In [22]:
# 9. FRESHNESS / VOLUME (row counts by month, gaps = missing batches)

monthly_volume = con.sql(f"""
    SELECT strftime(CAST(created_at AS TIMESTAMP), '%Y-%m') AS month,
           COUNT(*) AS n_rows
    FROM read_csv_auto('{path}')
    GROUP BY month
    ORDER BY month
""").df()
print("\n=== 7. ROW VOLUME BY MONTH ===")
print(monthly_volume)


=== 7. ROW VOLUME BY MONTH ===
      month  n_rows
0   2025-01     459
1   2025-02     436
2   2025-03     484
3   2025-04     497
4   2025-05     474
5   2025-06     473
6   2025-07     473
7   2025-08     532
8   2025-09     300
9   2025-10     401
10  2025-11     496
11  2025-12     285


In [26]:
# 10. TIME DRIFT: order_items.created_at vs parent orders.created_at

orders_path = "../../source_database/seed_data/orders.csv"

time_mismatch = con.sql(f"""
    SELECT oi.order_item_id, oi.order_id,
           oi.created_at AS item_created_at,
           o.created_at  AS order_created_at
    FROM read_csv_auto('{path}') oi
    JOIN read_csv_auto('{orders_path}') o ON oi.order_id = o.order_id
    WHERE oi.created_at != o.created_at
""").df()
print(f"\n=== 8. TIME DRIFT: item created_at != parent order created_at ===")
print(f"mismatched rows: {len(time_mismatch)}")

time_mismatch.head(10)


=== 8. TIME DRIFT: item created_at != parent order created_at ===
mismatched rows: 0


,order_item_id,order_id,item_created_at,order_created_at
